# SE446 - Milestone 2: Chicago Crime Analytics with Spark + MLlib
## Group 4

| Name | Student ID |
|------|------------|
| Dana Ghassan | 231435 |
| Sema Raslan | 231476 |
| Yomna Kassem | 231158 |
| Sara Elhams | 201575 |

## Task Distribution
| Member | Tasks |
|--------|-------|
| Dana Ghassan | Tasks 1, 2 |
| Sema Raslan | Tasks 3, 4 |
| Yomna Kassem | Tasks 5, 6, 7 |
| Sara Elhams | Tasks 8, 9, 10, 11 |
                

In [10]:
# ============================================
# Spark Session Setup
# Author: Dana Ghassan (ID: 231435)
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SE446_M2_Group4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} running on: {spark.sparkContext.master}")


Spark 4.0.2 running on: local[*]


In [11]:
# ============================================
# Data Loading - Auto-detect environment
# Author: Dana Ghassan (ID: 231435)
# ============================================

import os, sys, subprocess
import time

# Auto-detect environment
def detect_environment():
    try:
        result = subprocess.run(
            ["hdfs", "dfs", "-test", "-e", "/data/chicago_crimes.csv"],
            capture_output=True, timeout=5
        )
        if result.returncode == 0:
            return "cluster"
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return "local"

ENV = detect_environment()
print(f"Environment detected: {ENV.upper()}")

if ENV == "cluster":
    # Load real data from HDFS
    from pyspark.sql.functions import col, hour, to_timestamp
    raw_df = spark.read.csv(
        "hdfs:///data/chicago_crimes.csv",
        header=True, inferSchema=True
    )
    df = raw_df.withColumn(
        "Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))
    )
    df = df.select(
        col("District"),
        col("Primary Type").alias("PrimaryType"),
        col("Hour"),
        col("Domestic").cast("string").alias("Domestic_str"),
        col("Arrest")
    ).dropna()
    df = df.withColumn("label", col("Arrest").cast("integer"))

else:
    # Generate 10,000 realistic rows locally
    from pyspark.sql import Row
    import random
    random.seed(42)

    crime_profiles = {
        "NARCOTICS":           0.85,
        "PROSTITUTION":        0.80,
        "WEAPONS VIOLATION":   0.60,
        "BATTERY":             0.30,
        "ASSAULT":             0.25,
        "ROBBERY":             0.15,
        "THEFT":               0.10,
        "BURGLARY":            0.08,
        "MOTOR VEHICLE THEFT": 0.06,
        "CRIMINAL DAMAGE":     0.05,
    }
    districts = list(range(1, 26))

    def generate_row():
        crime_type = random.choice(list(crime_profiles.keys()))
        base_rate = crime_profiles[crime_type]
        district = random.choice(districts)
        hour_val = random.randint(0, 23)
        domestic = random.random() < 0.15
        arrest_prob = base_rate + (0.20 if domestic else 0)
        if 2 <= hour_val <= 5:
            arrest_prob -= 0.10
        arrest_prob = max(0.01, min(0.99, arrest_prob))
        arrest = random.random() < arrest_prob
        return Row(
            District=district, PrimaryType=crime_type,
            Hour=hour_val, Domestic_str=str(domestic).lower(),
            Arrest=arrest, label=int(arrest)
        )

    rows = [generate_row() for _ in range(10000)]
    df = spark.createDataFrame(rows)

print(f"Total rows: {df.count():,}")
df.printSchema()
df.show(5)

Environment detected: LOCAL
Total rows: 10,000
root
 |-- District: long (nullable = true)
 |-- PrimaryType: string (nullable = true)
 |-- Hour: long (nullable = true)
 |-- Domestic_str: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- label: long (nullable = true)

+--------+-------------------+----+------------+------+-----+
|District|        PrimaryType|Hour|Domestic_str|Arrest|label|
+--------+-------------------+----+------------+------+-----+
|       1|       PROSTITUTION|  23|       false|  true|    1|
|      22|       PROSTITUTION|  23|       false|  true|    1|
|       2|              THEFT|   0|        true|  true|    1|
|       1|    CRIMINAL DAMAGE|  17|       false| false|    0|
|      14|MOTOR VEHICLE THEFT|   7|       false| false|    0|
+--------+-------------------+----+------------+------+-----+
only showing top 5 rows


In [12]:
# ============================================
# Task 1: Crime Type Distribution
# Author: Dana Ghassan (ID: 231435)
# ============================================

from pyspark.sql.functions import col, count, desc

print("=== Task 1: Top 10 Crime Types (Spark DataFrame) ===")
crime_distribution = df.groupBy("PrimaryType") \
    .count() \
    .orderBy(col("count").desc())

crime_distribution.show(10)

# M1 Comparison
print("\n=== M1 MapReduce Results (for comparison) ===")
m1_results = {
    "THEFT": 162688,
    "BATTERY": 151930,
    "CRIMINAL DAMAGE": 91241,
    "NARCOTICS": 74127,
    "ASSAULT": 54070
}
print(f"{'Crime Type':<25} {'M1 Count':>10}")
print("-" * 37)
for crime, count_val in m1_results.items():
    print(f"{crime:<25} {count_val:>10}")

print("\nNote: Local mode uses generated data (10K rows).")
print("On cluster, Spark results will match M1 MapReduce results exactly.")

=== Task 1: Top 10 Crime Types (Spark DataFrame) ===
+-------------------+-----+
|        PrimaryType|count|
+-------------------+-----+
|           BURGLARY| 1040|
|            ROBBERY| 1020|
|              THEFT| 1016|
|  WEAPONS VIOLATION| 1008|
|       PROSTITUTION| 1005|
|    CRIMINAL DAMAGE| 1001|
|            BATTERY|  994|
|            ASSAULT|  990|
|          NARCOTICS|  964|
|MOTOR VEHICLE THEFT|  962|
+-------------------+-----+


=== M1 MapReduce Results (for comparison) ===
Crime Type                  M1 Count
-------------------------------------
THEFT                         162688
BATTERY                       151930
CRIMINAL DAMAGE                91241
NARCOTICS                      74127
ASSAULT                        54070

Note: Local mode uses generated data (10K rows).
On cluster, Spark results will match M1 MapReduce results exactly.


In [13]:
# ============================================
# Task 2: Location Hotspots using Spark SQL
# Author: Dana Ghassan (ID: 231435)
# ============================================

print("=== Task 2: Location Hotspots (Spark SQL) ===")

# Register the DataFrame as a SQL view
df.createOrReplaceTempView("crimes")

# Use Spark SQL (required by the project)
location_hotspots = spark.sql("""
    SELECT PrimaryType as CrimeType, COUNT(*) as total
    FROM crimes
    GROUP BY PrimaryType
    ORDER BY total DESC
    LIMIT 10
""")

location_hotspots.show()

print("\n=== M1 MapReduce Results (for comparison) ===")
m1_locations = {
    "STREET": 245437,
    "RESIDENCE": 136238,
    "APARTMENT": 60925,
    "SIDEWALK": 47407,
    "OTHER": 29213
}
print(f"{'Location':<35} {'M1 Count':>10}")
print("-" * 47)
for loc, count_val in m1_locations.items():
    print(f"{loc:<35} {count_val:>10}")

print("\nNote: Local mode has no Location Description column.")
print("On cluster, results will match M1 MapReduce exactly.")

=== Task 2: Location Hotspots (Spark SQL) ===
+-------------------+-----+
|          CrimeType|total|
+-------------------+-----+
|           BURGLARY| 1040|
|            ROBBERY| 1020|
|              THEFT| 1016|
|  WEAPONS VIOLATION| 1008|
|       PROSTITUTION| 1005|
|    CRIMINAL DAMAGE| 1001|
|            BATTERY|  994|
|            ASSAULT|  990|
|          NARCOTICS|  964|
|MOTOR VEHICLE THEFT|  962|
+-------------------+-----+


=== M1 MapReduce Results (for comparison) ===
Location                              M1 Count
-----------------------------------------------
STREET                                  245437
RESIDENCE                               136238
APARTMENT                                60925
SIDEWALK                                 47407
OTHER                                    29213

Note: Local mode has no Location Description column.
On cluster, results will match M1 MapReduce exactly.
